## **2 кейс**

**Выгрузка активности с ItResume**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [1]:
!wget https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv

--2026-04-29 07:03:28--  https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.github.com (gist.github.com)... 140.82.121.4
Connecting to gist.github.com (gist.github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv [following]
--2026-04-29 07:03:28--  https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215378 (210K) [text/plain]
Saving to: ‘codesubmit.csv’

codesubmit.csv      100%[===================>] 210.33K  --.-KB/s    in 0.01s   

2026-04-29 07:03:28 (17.6 MB/s) - ‘co

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [2]:
import pandas as pd

df = pd.read_csv('codesubmit.csv', sep = ';')
df

,created_at,user_id,problem_id,is_correct,type
0,2023-04-30 13:47:14.344471,7,870,1.0,submit
1,2023-04-30 13:46:15.949925,7,870,0.0,submit
2,2023-04-30 16:13:26.005286,173,21,1.0,submit
3,2023-04-30 16:13:06.739782,173,21,NaN,run
4,2023-04-30 15:52:00.195532,173,25,1.0,submit
...,...,...,...,...,...
4994,2023-04-30 21:52:00.269123,13493,435,NaN,run
4995,2023-04-30 21:51:01.094234,13493,435,1.0,submit
4996,2023-04-30 21:50:52.059690,13493,435,NaN,run
4997,2023-04-30 21:42:24.323689,13493,1086,NaN,run


### **Решения**

#### **Задача 1**

Ваша задача - выяснить сколько в среднем тратится времени на решение задачи.

**Примечание**: для правильного подсчета - рассчитайте сначала среднее время решения по каждой задаче в отдельности, и только затем находите общее среднее время решения задач.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res`.


**Решение**

Напишите свое решение ниже

In [4]:

# Ваше решение здесь

import pandas as pd

df = pd.read_csv('codesubmit.csv', sep=';')
df['created_at'] = pd.to_datetime(df['created_at'])

data = df.sort_values('created_at')

first_attempt = (
    data.groupby(['user_id', 'problem_id'])['created_at']
    .min()
    .reset_index(name='first_attempt')
)

first_correct = (
    data[data['is_correct'] == 1]
    .groupby(['user_id', 'problem_id'])['created_at']
    .min()
    .reset_index(name='first_correct')
)

merged = first_attempt.merge(first_correct, on=['user_id', 'problem_id'])

merged['time_spent'] = (
    merged['first_correct'] - merged['first_attempt']
).dt.total_seconds()

merged = merged[merged['time_spent'] > 0]

problem_avg = merged.groupby('problem_id')['time_spent'].mean()

res = round(problem_avg.mean(), 2)
res

np.float64(611.86)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [5]:
try:
    assert res == 611.86
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача - выяснить сколько часов в среднем проводит юзер в день на платформе. Перерывы в активности за день - не учитываем.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res2`.

**Решение**

Напишите свое решение ниже

In [8]:

# Ваше решение здесь

import pandas as pd

df = pd.read_csv('codesubmit.csv', sep=';')
df['created_at'] = pd.to_datetime(df['created_at'])

df['date'] = df['created_at'].dt.date

user_day_time = (
    df.groupby(['user_id', 'date'])['created_at']
    .agg(['min', 'max'])
    .reset_index()
)

user_day_time['time_spent'] = (
    user_day_time['max'] - user_day_time['min']
).dt.total_seconds() / 3600

res2 = round(user_day_time['time_spent'].mean(), 2)
res2

np.float64(1.7)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [9]:
try:
    assert res2 == 1.7
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Теперь давайте посмотрим на активные сеансы. Выясните, сколько задач в среднем решается за один активный сеанс.

**Активный сеанс** - период, когда между любой активностью пользователя разница менее или равна часу, не более

**Важно**: в расчет берем не только успешные попытки решений (`is_correct=1`), а и неуспешные тоже (`is_correct=0`), и тип `run` в том числе.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res3`.

**Решение**

Напишите свое решение ниже

In [10]:

# Ваше решение здесь

data = df.sort_values(['user_id', 'created_at']).copy()

data['diff'] = (
    data.groupby('user_id')['created_at']
    .diff()
    .dt.total_seconds()
)

data['new_session'] = ((data['diff'].isna()) | (data['diff'] > 3600)).astype(int)

data['session_id'] = data.groupby('user_id')['new_session'].cumsum()

tasks_per_session = (
    data.groupby(['user_id', 'session_id'])['problem_id']
    .nunique()
)

res3 = round(tasks_per_session.mean(), 2)
res3

np.float64(3.14)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [11]:
try:
    assert res3 == 3.14
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 4**

И финальная - найдите самый "популярный" час дня на нашей платформе.

Популярность определяем максимальным количеством уникальных пользователей, совершающих какую-либо активность в этот период

Результат в числовом формате запишите в переменную `res4`.

Например, самым популярным часом стал период с 22 до 23, тогда в переменной `res4` должно лежать **22**. Обозначающее начало этого периода.

**Решение**

Напишите свое решение ниже

In [12]:

# Ваше решение здесь

df['hour'] = df['created_at'].dt.hour

res4 = df.groupby('hour')['user_id'].nunique().idxmax()
res4

np.int32(16)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [13]:
try:
    assert res4 == 16
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
